# VARIANCES

This notebook is dedicated at the empirical study of how the variance of the errors vary among different horizons.

In [12]:
from src.data_handler import *
from src.config_files import *
from src.direct_models import *
from src.mheme import *
from src.plot_handler import *

import os

In [ ]:
WINDOW = 168
HORIZON = 24

DATA_PATH = '../data'

MODEL_PATH = '../models'

TCN_PATH_CONFIG_LOAD = '../src/config_files/tcn_config.json'
TCN_PATH_SAVE = '../models/tcn'

XGB_PATH_CONFIG_LOAD = '../src/config_files/xgb_config.json'
XGB_PATH_SAVE = '../models/xgb'

In [14]:
X, data = data_loader(data_path = DATA_PATH, dataset = 'traffic')
X = (X - np.mean(X))/np.std(X)

In [15]:
kwargs = {"name" : "Traffic", "color" : "blue", "title" : "Traffic volume over time", "x_axis" : "Time", "y_axis" : "Traffic volume"}
plot_time_series(X[-500:], **kwargs)

In [16]:
X_slide, y_slide = sliding_window(X, window=WINDOW, horizon=HORIZON, k = 17)
train, val, test = train_validation_test_split(X_slide, y_slide)

In [17]:
#umheme_tcn = UMHEMe.load_model(os.path.join(MODEL_PATH, 'tcn_traffic_2.pkl'))
umheme_tcn = UMHEMe(HORIZON, WINDOW, model_class=TCN, config_path=TCN_PATH_CONFIG_LOAD, skip=4)
umheme_tcn.fit(train[0], train[1])

Fitting class : <class 'src.direct_models.TCN'>; horizon : 1


Training TCN: 100%|██████████| 150/150 [00:05<00:00, 28.71it/s]


Fitting class : <class 'src.direct_models.TCN'>; horizon : 5


Training TCN: 100%|██████████| 150/150 [00:04<00:00, 33.55it/s]


Fitting class : <class 'src.direct_models.TCN'>; horizon : 9


...

Training TCN: 100%|██████████| 150/150 [00:04<00:00, 33.35it/s]


Fitting class : <class 'src.direct_models.TCN'>; horizon : 41


Training TCN: 100%|██████████| 150/150 [00:04<00:00, 33.09it/s]


Fitting class : <class 'src.direct_models.TCN'>; horizon : 45


Training TCN: 100%|██████████| 150/150 [00:04<00:00, 34.13it/s]


In [18]:
umheme_tcn.save_model(os.path.join(MODEL_PATH, 'tcn_traffic_2.pkl'))

In [19]:
umheme_tcn.compute_weights(train[0], train[1])

In [20]:
umheme_tcn.visualize_variances(None)
umheme_tcn.visualize_errors(None)
umheme_tcn.visualize_weights(None)

In [21]:
preds = umheme_tcn.whole_predict(test[0])
ens_preds = umheme_tcn.predict(test[0])
for i in range (10):
    pred = {model : preds[model][i] for model in preds}
    plot_multiple_forecast(test[0][i], test[1][i], pred, **kwargs)
    errs = np.array([mse(preds[model][i], test[1][i]) for model in preds])
    avg_mse = np.mean(errs)
    min_mse = np.min(errs)
    max_mse = np.max(errs)
    ensemble_mse = mse(ens_preds[i], test[1][i])
    print(f"min mse : {min_mse}\nmax mse : {max_mse}\naverage mse : {avg_mse}\nensemble mse : {ensemble_mse}")
    

min mse : 0.14419324696063995
max mse : 1.2559062242507935
average mse : 0.38144609332084656
ensemble mse : 0.1630687117576599


min mse : 0.24192140996456146
max mse : 0.7129077911376953
average mse : 0.36945071816444397
ensemble mse : 0.25667473673820496


min mse : 0.6427973508834839
max mse : 1.8068723678588867
average mse : 0.9428377151489258
ensemble mse : 0.8515452742576599


min mse : 0.26054826378822327
max mse : 1.6921027898788452
average mse : 0.553298830986023
ensemble mse : 0.34752774238586426


min mse : 2.8192481994628906
max mse : 4.455517292022705
average mse : 3.1400091648101807
ensemble mse : 2.9738967418670654


min mse : 4.123548984527588
max mse : 5.104908466339111
average mse : 4.367499828338623
ensemble mse : 4.25626802444458


min mse : 1.7571274042129517
max mse : 5.019413471221924
average mse : 3.255450963973999
ensemble mse : 2.2533655166625977


min mse : 1.570151925086975
max mse : 2.581012487411499
average mse : 1.8257068395614624
ensemble mse : 1.6314672231674194


min mse : 0.5305799841880798
max mse : 1.4798187017440796
average mse : 0.6862800717353821
ensemble mse : 0.5940105319023132


min mse : 0.5348835587501526
max mse : 1.2853175401687622
average mse : 0.8272168040275574
ensemble mse : 0.7619768977165222


In [22]:
preds = umheme_tcn.whole_predict(test[0])
ens_preds = umheme_tcn.predict(test[0])
for i in range (10):
    plot_forecast(test[0][i], test[1][i], ens_preds[i], **kwargs)